# Workflow Completo: Modelagem Multivariada e Gestao de Risco

Este notebook demonstra o pipeline completo de modelagem multivariada de volatilidade
integrada com gestao de risco de portfolio.

## Pipeline

1. **Dados Multivariados** - Portfolio de 4 FX pairs
2. **GARCH Univariado** - Modelagem individual de cada serie
3. **Modelagem Multivariada** - CCC e DCC
4. **Correlacoes Dinamicas** - Evolucao temporal das correlacoes
5. **Portfolio de Minima Variancia** - Otimizacao com covariancia condicional
6. **VaR do Portfolio** - Risco agregado
7. **Backtesting** - Validacao out-of-sample
8. **Analise de Regime** - MS-GARCH para regimes de crise
9. **Stress Testing** - VaR condicional a regimes
10. **Report Final** - Consolidacao dos resultados

**Dados**: FX majors (EUR/USD, GBP/USD, USD/JPY, USD/CHF)

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings('ignore')

# archbox imports
from archbox.models.garch import GARCH
from archbox.multivariate.ccc import CCC
from archbox.multivariate.dcc import DCC
from archbox.multivariate.portfolio import minimum_variance_portfolio
from archbox.regime.ms_garch import MS_GARCH
from archbox.risk.var import christoffersen_test, kupiec_test

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Imports carregados com sucesso.')

## Etapa 1: Dados Multivariados

Carregamos retornos diarios de 4 pares FX majors:
- **EUR/USD**: Euro vs Dolar
- **GBP/USD**: Libra vs Dolar
- **USD/JPY**: Dolar vs Iene
- **USD/CHF**: Dolar vs Franco Suico

Estes pares sao altamente liquidos e apresentam correlacoes dinamicas interessantes.

In [ ]:
# TODO: Carregue fx_majors.csv, visualize correlacoes e volatilidades

data_path = '../data/fx_majors.csv'
df = pd.read_csv(data_path, index_col=0, parse_dates=True)
returns = df.dropna()
pairs = returns.columns.tolist()

print(f'Periodo: {returns.index[0]} a {returns.index[-1]}')
print(f'Observacoes: {len(returns)}')
print(f'Pares FX: {pairs}')

print('\nEstatisticas descritivas:')
print(returns.describe().round(6))

# Plotar series
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for i, (ax, col) in enumerate(zip(axes.flat, pairs, strict=False)):
    ax.plot(returns.index, returns[col].values, linewidth=0.5)
    ax.set_title(f'Retornos: {col}')
    ax.axhline(y=0, color='r', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Matriz de correlacao incondicional
print('\nMatriz de correlacao incondicional:')
print(returns.corr().round(4))

## Etapa 2: GARCH Univariado para cada serie

Antes da modelagem multivariada, estimamos GARCH(1,1) individualmente
para cada par FX. As volatilidades condicionais univariadas serao usadas
como entrada para os modelos DCC e CCC.

In [ ]:
# TODO: Estime GARCH(1,1) para cada par FX individualmente

univariate_results = {}

for pair in pairs:
    print(f'Estimando GARCH(1,1) para {pair}...')
    model = GARCH(p=1, q=1)
    result = model.fit(returns[pair].values)
    univariate_results[pair] = result
    print(f'  omega={result.params[0]:.6f}, alpha={result.params[1]:.4f}, beta={result.params[2]:.4f}')
    print(f'  Persistencia (alpha+beta): {result.params[1] + result.params[2]:.4f}')

# Plotar volatilidades condicionais
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for i, (ax, pair) in enumerate(zip(axes.flat, pairs, strict=False)):
    vol = univariate_results[pair].conditional_volatility
    ax.plot(returns.index, vol, linewidth=0.7)
    ax.set_title(f'Volatilidade Condicional: {pair}')
    ax.set_ylabel('sigma_t')
plt.tight_layout()
plt.show()

## Etapa 3: Modelagem Multivariada

Estimamos dois modelos multivariados de volatilidade:

- **CCC (Constant Conditional Correlation)**: Correlacoes fixas no tempo
- **DCC (Dynamic Conditional Correlation)**: Correlacoes variam ao longo do tempo

Comparamos os modelos por AIC para determinar se as correlacoes dinamicas
sao estatisticamente relevantes.

In [ ]:
# TODO: Estime CCC e DCC, compare AIC

returns_matrix = returns.values

# CCC
print('Estimando CCC...')
ccc_model = CCC()
ccc_result = ccc_model.fit(returns_matrix)
print(f'  Log-Likelihood: {ccc_result.loglikelihood:.4f}')
print(f'  AIC: {ccc_result.aic:.4f}')

# DCC
print('\nEstimando DCC...')
dcc_model = DCC()
dcc_result = dcc_model.fit(returns_matrix)
print(f'  DCC params: a={dcc_result.dcc_a:.4f}, b={dcc_result.dcc_b:.4f}')
print(f'  Log-Likelihood: {dcc_result.loglikelihood:.4f}')
print(f'  AIC: {dcc_result.aic:.4f}')

# Comparacao
print('\n=== Comparacao CCC vs DCC ===')
comp = pd.DataFrame({
    'Modelo': ['CCC', 'DCC'],
    'Log-Likelihood': [ccc_result.loglikelihood, dcc_result.loglikelihood],
    'AIC': [ccc_result.aic, dcc_result.aic],
    'BIC': [ccc_result.bic, dcc_result.bic]
})
print(comp.to_string(index=False))

best_mv = 'DCC' if dcc_result.aic < ccc_result.aic else 'CCC'
print(f'\nModelo selecionado: {best_mv}')

## Etapa 4: Correlacoes Dinamicas

Extraimos as correlacoes condicionais do modelo DCC e visualizamos como
as correlacoes entre os pares FX evoluem ao longo do tempo.

Correlacoes dinamicas sao fundamentais para:
- Rebalanceamento de portfolio
- Identificacao de periodos de contagio
- Hedge ratio dinamico

In [ ]:
# TODO: Extraia e plote correlacoes condicionais do DCC

# Extrair correlacoes condicionais do DCC
R_t = dcc_result.dynamic_correlations  # (T, k, k)

# Plotar correlacoes selecionadas
n_pairs = len(pairs)
pair_combos = [(i, j) for i in range(n_pairs) for j in range(i+1, n_pairs)]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for idx, (i, j) in enumerate(pair_combos):
    ax = axes.flat[idx]
    corr_ij = R_t[:, i, j]
    ax.plot(returns.index, corr_ij, linewidth=0.7)
    ax.set_title(f'Correlacao dinamica: {pairs[i]} vs {pairs[j]}')
    ax.set_ylabel('rho_t')
    # Correlacao incondicional como referencia
    unconditional = returns[pairs[i]].corr(returns[pairs[j]])
    ax.axhline(y=unconditional, color='r', linestyle='--', alpha=0.5, label=f'Incondicional: {unconditional:.3f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('As correlacoes dinamicas mostram variacao significativa ao longo do tempo.')
print('Periodos de estresse tendem a apresentar correlacoes mais altas (contagio).')

## Etapa 5: Portfolio de Minima Variancia

Usando a matriz de covariancia condicional H_t do DCC, calculamos os pesos
do **Minimum Variance Portfolio (MVP)** em cada instante t:

$$w_t = \frac{H_t^{-1} \mathbf{1}}{\mathbf{1}' H_t^{-1} \mathbf{1}}$$

Os pesos variam dinamicamente conforme a estrutura de volatilidade e correlacao muda.

In [ ]:
# TODO: Calcule pesos do MVP usando H_t do DCC, plote evolucao dos pesos

H_t = dcc_result.covariance_matrices  # (T, k, k)

# Calcular pesos do MVP para cada t
weights = minimum_variance_portfolio(H_t)

# DataFrame de pesos
weights_df = pd.DataFrame(weights, index=returns.index, columns=pairs)

print('=== Pesos do Portfolio de Minima Variancia ===')
print('\nPesos medios:')
print(weights_df.mean().round(4))
print('\nPesos no ultimo dia:')
print(weights_df.iloc[-1].round(4))

# Plotar evolucao dos pesos
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Pesos ao longo do tempo
for pair in pairs:
    axes[0].plot(weights_df.index, weights_df[pair], linewidth=0.7, label=pair)
axes[0].set_title('Evolucao dos Pesos do MVP (DCC)')
axes[0].set_ylabel('Peso')
axes[0].legend()
axes[0].axhline(y=1/len(pairs), color='gray', linestyle=':', alpha=0.5, label='Equal weight')

# Area chart (stacked)
axes[1].stackplot(weights_df.index, [weights_df[p] for p in pairs],
                  labels=pairs, alpha=0.7)
axes[1].set_title('Composicao do Portfolio ao Longo do Tempo')
axes[1].set_ylabel('Peso acumulado')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

## Etapa 6: VaR do Portfolio

Calculamos o Value-at-Risk do portfolio usando a covariancia condicional do DCC.
A variancia do portfolio no tempo t e:

$$\sigma_{p,t}^2 = w_t' H_t w_t$$

O VaR parametrico assume normalidade condicional:
$$\text{VaR}_{\alpha,t} = \mu_{p,t} + z_{\alpha} \cdot \sigma_{p,t}$$

In [ ]:
# TODO: Calcule VaR(95%) do portfolio usando covariancia condicional

# Retornos do portfolio
portfolio_returns = np.sum(returns.values * weights, axis=1)

# Variancia do portfolio: w' H_t w
T = len(portfolio_returns)
portfolio_var = np.zeros(T)
for t in range(T):
    w = weights[t]
    portfolio_var[t] = w @ H_t[t] @ w
portfolio_vol = np.sqrt(portfolio_var)

# VaR parametrico
z_95 = stats.norm.ppf(0.05)
z_99 = stats.norm.ppf(0.01)
var_port_95 = np.mean(portfolio_returns) + z_95 * portfolio_vol
var_port_99 = np.mean(portfolio_returns) + z_99 * portfolio_vol

# ES
es_port_95 = np.mean(portfolio_returns) + portfolio_vol * stats.norm.pdf(z_95) / 0.05
es_port_99 = np.mean(portfolio_returns) + portfolio_vol * stats.norm.pdf(z_99) / 0.01

# Plotar
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(returns.index, portfolio_returns, 'b-', linewidth=0.5, alpha=0.7, label='Retornos portfolio')
ax.plot(returns.index, var_port_95, 'r-', linewidth=0.8, label='VaR 95%')
ax.plot(returns.index, var_port_99, 'darkred', linewidth=0.8, label='VaR 99%')
ax.set_title('VaR do Portfolio FX (DCC-MVP)')
ax.set_ylabel('Retorno / VaR')
ax.legend()
plt.tight_layout()
plt.show()

violations_95 = np.sum(portfolio_returns < var_port_95)
violations_99 = np.sum(portfolio_returns < var_port_99)
print(f'Violacoes VaR 95%: {violations_95} ({100*violations_95/T:.2f}%, esperado ~5%)')
print(f'Violacoes VaR 99%: {violations_99} ({100*violations_99/T:.2f}%, esperado ~1%)')

## Etapa 7: Backtesting do VaR do Portfolio

Aplicamos testes formais de backtesting ao VaR do portfolio:

- **Kupiec (POF)**: Frequencia de violacoes consistente com o nivel de confianca?
- **Christoffersen (CC)**: Violacoes sao independentes (sem clustering)?

In [ ]:
# TODO: Execute Kupiec e Christoffersen no VaR do portfolio

# Split out-of-sample (80/20)
split = int(T * 0.8)
ret_oos = portfolio_returns[split:]
var_oos_95 = var_port_95[split:]
var_oos_99 = var_port_99[split:]

# Backtesting VaR 95%
print('=== Backtesting VaR 95% (portfolio, out-of-sample) ===')
hits_95 = (ret_oos < var_oos_95).astype(int)
kup_95 = kupiec_test(hits_95, alpha=0.05)
cc_95 = christoffersen_test(hits_95, alpha=0.05)
print(f'  Violacoes: {hits_95.sum()} de {len(hits_95)} ({100*hits_95.mean():.2f}%)')
print(f'  Kupiec: stat={kup_95.statistic:.4f}, p={kup_95.pvalue:.4f}')
print(f'  Christoffersen: stat={cc_95.statistic:.4f}, p={cc_95.pvalue:.4f}')

print()

# Backtesting VaR 99%
print('=== Backtesting VaR 99% (portfolio, out-of-sample) ===')
hits_99 = (ret_oos < var_oos_99).astype(int)
kup_99 = kupiec_test(hits_99, alpha=0.01)
cc_99 = christoffersen_test(hits_99, alpha=0.01)
print(f'  Violacoes: {hits_99.sum()} de {len(hits_99)} ({100*hits_99.mean():.2f}%)')
print(f'  Kupiec: stat={kup_99.statistic:.4f}, p={kup_99.pvalue:.4f}')
print(f'  Christoffersen: stat={cc_99.statistic:.4f}, p={cc_99.pvalue:.4f}')

print('\np-valor > 0.05 indica que o VaR passou no backtest.')

## Etapa 8: Analise de Regime

Aplicamos um modelo **Markov-Switching GARCH** com 2 regimes na volatilidade
do portfolio para identificar:

- **Regime 1**: Baixa volatilidade (mercado tranquilo)
- **Regime 2**: Alta volatilidade (regime de crise)

A transicao entre regimes e governada por uma cadeia de Markov com
probabilidades de transicao estimadas dos dados.

In [ ]:
# TODO: Estime MS(2)-GARCH no retorno do portfolio, identifique regimes de crise

print('Estimando MS(2)-GARCH no retorno do portfolio...')
ms_model = MS_GARCH(n_regimes=2, p=1, q=1)
ms_result = ms_model.fit(portfolio_returns)

# Parametros por regime
print('\n=== Parametros MS-GARCH ===')
for regime in range(2):
    print(f'\nRegime {regime + 1}:')
    print(f'  omega: {ms_result.regime_params[regime]["omega"]:.6f}')
    print(f'  alpha: {ms_result.regime_params[regime]["alpha"]:.4f}')
    print(f'  beta: {ms_result.regime_params[regime]["beta"]:.4f}')

# Probabilidades de transicao
print('\nMatriz de transicao:')
print(pd.DataFrame(ms_result.transition_matrix,
                    index=['De Regime 1', 'De Regime 2'],
                    columns=['Para Regime 1', 'Para Regime 2']).round(4))

# Probabilidades suavizadas
smoothed_probs = ms_result.smoothed_probabilities

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Retornos coloridos por regime
regime_2_prob = smoothed_probs[:, 1]
axes[0].plot(returns.index, portfolio_returns, 'b-', linewidth=0.5, alpha=0.5)
axes[0].fill_between(returns.index, portfolio_returns.min(), portfolio_returns.max(),
                      where=regime_2_prob > 0.5, color='red', alpha=0.2, label='Regime crise')
axes[0].set_title('Retornos do Portfolio com Regimes')
axes[0].legend()

# Probabilidade de regime de crise
axes[1].plot(returns.index, regime_2_prob, 'r-', linewidth=0.7)
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('Probabilidade de Regime de Crise (Regime 2)')
axes[1].set_ylabel('P(Regime 2)')

plt.tight_layout()
plt.show()

# Periodos em regime de crise
crisis_pct = 100 * np.mean(regime_2_prob > 0.5)
print(f'\nPercentual do tempo em regime de crise: {crisis_pct:.1f}%')

## Etapa 9: Stress Testing

Utilizamos as informacoes de regime para calcular o VaR **condicional ao
regime de crise** (Regime 2). Este VaR de stress e mais conservador,
refletindo a volatilidade elevada em periodos de turbulencia.

$$\text{VaR}_{\text{stress}} = \mu + z_\alpha \cdot \sigma_{\text{Regime 2}}$$

In [ ]:
# TODO: Calcule VaR condicional ao regime de crise (regime 2)

# Volatilidade condicional por regime
vol_regime1 = ms_result.conditional_volatility_regime(0)
vol_regime2 = ms_result.conditional_volatility_regime(1)

print('=== Stress Testing: VaR Condicional ao Regime ===')
print(f'\nVolatilidade media - Regime 1 (tranquilo): {np.mean(vol_regime1):.6f}')
print(f'Volatilidade media - Regime 2 (crise): {np.mean(vol_regime2):.6f}')
print(f'Ratio crise/tranquilo: {np.mean(vol_regime2)/np.mean(vol_regime1):.2f}x')

# VaR de stress (usando volatilidade do regime de crise)
mu_port = np.mean(portfolio_returns)
var_stress_95 = mu_port + z_95 * np.mean(vol_regime2)
var_stress_99 = mu_port + z_99 * np.mean(vol_regime2)
var_normal_95 = mu_port + z_95 * np.mean(portfolio_vol)
var_normal_99 = mu_port + z_99 * np.mean(portfolio_vol)

stress_table = pd.DataFrame({
    'Cenario': ['Normal', 'Estresse (Regime 2)'],
    'VaR 95%': [var_normal_95, var_stress_95],
    'VaR 99%': [var_normal_99, var_stress_99]
})
print('\nComparacao VaR Normal vs Estresse:')
print(stress_table.to_string(index=False))

# Plotar comparacao
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(2)
width = 0.35
ax.bar(x - width/2, [abs(var_normal_95), abs(var_stress_95)], width, label='VaR 95%', color='steelblue')
ax.bar(x + width/2, [abs(var_normal_99), abs(var_stress_99)], width, label='VaR 99%', color='darkred')
ax.set_xticks(x)
ax.set_xticklabels(['Normal', 'Estresse (Regime Crise)'])
ax.set_ylabel('|VaR| (perda absoluta)')
ax.set_title('Stress Testing: VaR Normal vs Regime de Crise')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nO VaR de estresse e {abs(var_stress_99)/abs(var_normal_99):.1f}x mais conservador que o VaR normal.')

## Etapa 10: Report Final

Consolidamos todos os resultados do workflow multivariado em um report abrangente.

In [ ]:
# TODO: Gere report consolidado com tabelas e graficos-chave

print('=' * 70)
print('REPORT FINAL: WORKFLOW MULTIVARIADO E GESTAO DE RISCO')
print('=' * 70)

# 1. Dados
print('\n--- 1. Dados ---')
print(f'Ativos: {pairs}')
print(f'Observacoes: {len(returns)}')
print(f'Periodo: {returns.index[0]} a {returns.index[-1]}')

# 2. Modelos univariados
print('\n--- 2. GARCH Univariados ---')
for pair in pairs:
    res = univariate_results[pair]
    persist = res.params[1] + res.params[2]
    print(f'  {pair}: alpha={res.params[1]:.4f}, beta={res.params[2]:.4f}, persistencia={persist:.4f}')

# 3. Modelo multivariado
print('\n--- 3. Modelo Multivariado ---')
print(comp.to_string(index=False))
print(f'Selecionado: {best_mv}')

# 4. Portfolio
print('\n--- 4. Portfolio de Minima Variancia ---')
print('Pesos medios:')
for pair in pairs:
    print(f'  {pair}: {weights_df[pair].mean():.4f}')
print(f'Retorno medio do portfolio: {np.mean(portfolio_returns)*252:.4f} (anualizado)')
print(f'Volatilidade media do portfolio: {np.mean(portfolio_vol)*np.sqrt(252):.4f} (anualizada)')

# 5. Risco
print('\n--- 5. Medidas de Risco do Portfolio ---')
print(f'VaR 95% (medio): {np.mean(var_port_95):.6f}')
print(f'VaR 99% (medio): {np.mean(var_port_99):.6f}')
print(f'ES 95% (medio): {np.mean(es_port_95):.6f}')
print(f'ES 99% (medio): {np.mean(es_port_99):.6f}')

# 6. Backtesting
print('\n--- 6. Backtesting (out-of-sample) ---')
print(f'VaR 95%: Kupiec p={kup_95.pvalue:.4f}, Christoffersen p={cc_95.pvalue:.4f}')
print(f'VaR 99%: Kupiec p={kup_99.pvalue:.4f}, Christoffersen p={cc_99.pvalue:.4f}')

# 7. Regimes
print('\n--- 7. Analise de Regime ---')
print(f'Regime de crise: {crisis_pct:.1f}% do tempo')
print(f'Vol ratio (crise/normal): {np.mean(vol_regime2)/np.mean(vol_regime1):.2f}x')

# 8. Stress testing
print('\n--- 8. Stress Testing ---')
print(stress_table.to_string(index=False))

# Summary consolidado
print('\n--- Resumo Executivo ---')
summary = pd.DataFrame({
    'Metrica': [
        'Modelo multivariado', 'DCC a', 'DCC b',
        'VaR 99% (normal)', 'VaR 99% (estresse)',
        'Kupiec 95% p-valor', 'Christoffersen 95% p-valor',
        'Regime crise (%)', 'Vol ratio crise/normal'
    ],
    'Valor': [
        best_mv, f'{dcc_result.dcc_a:.4f}', f'{dcc_result.dcc_b:.4f}',
        f'{np.mean(var_port_99):.6f}', f'{var_stress_99:.6f}',
        f'{kup_95.pvalue:.4f}', f'{cc_95.pvalue:.4f}',
        f'{crisis_pct:.1f}', f'{np.mean(vol_regime2)/np.mean(vol_regime1):.2f}'
    ]
})
print(summary.to_string(index=False))

print('\n' + '=' * 70)
print('Pipeline multivariado completo executado com sucesso!')
print('=' * 70)